# 03 训练工程：让训练可控、可断、可复现

> 前置：`04-neural-networks/03-training-loop`、`04-neural-networks/06-optimizers-in-practice`。
> 目标：从"能训练的循环"升级为"能交付的训练系统"。本课用 **numpy 手写一个两层网络**（不依赖任何框架），把训练工程的五个护栏逐个加上并验证效果——工程细节看得见、摸得着。

## 训练一个模型需要哪些护栏

| 护栏 | 解决的问题 |
|------|-----------|
| **seed 管理** | 复现：同一份代码两次训练结果必须一致 |
| **checkpoint** | 意外中断/算力预算：随时能断点续训 |
| **early stopping** | 防过拟合：验证集不涨就停，省算力 |
| **学习率调度** | 收敛质量：前期大步探索、后期小步精修 |
| **梯度裁剪** | 稳定性：防止梯度爆炸把参数冲飞 |

真实项目（如训练 7B LLM）里，这些护栏决定训练**能不能跑完**，而不只是好不好。

In [ ]:
# 本模块通用导入（全部 CPU 即可运行）
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

# 中文字体兼容（Windows / macOS）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# 二分类玩具数据（月亮形，非线性可分）
X, y = make_moons(n_samples=800, noise=0.15, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("训练", X_tr.shape, "验证", X_va.shape)

class Net:
    """两层 MLP：d_in -> 32(ReLU) -> 1(sigmoid)。手写前向/反向。"""
    def __init__(self, d_in=2, d_h=32, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.standard_normal((d_in, d_h)) * 0.5
        self.b1 = np.zeros(d_h)
        self.W2 = rng.standard_normal((d_h, 1)) * 0.5
        self.b2 = np.zeros(1)
    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = np.maximum(self.z1, 0)            # ReLU
        self.z2 = self.a1 @ self.W2 + self.b2
        return 1.0 / (1.0 + np.exp(-self.z2))       # sigmoid
    def backward(self, X, y, p):
        m = X.shape[0]
        dz2 = (p - y.reshape(-1, 1)) / m            # BCE 的梯度
        dW2 = self.a1.T @ dz2;  db2 = dz2.sum(0)
        dz1 = (dz2 @ self.W2.T) * (self.z1 > 0)
        dW1 = X.T @ dz1;        db1 = dz1.sum(0)
        return dW1, db1, dW2, db2
    def params(self):
        return {"W1": self.W1, "b1": self.b1, "W2": self.W2, "b2": self.b2}
    def load(self, p):
        for k, v in p.items():
            getattr(self, k)[...] = v

def acc(net, X, y):
    return ((net.forward(X) > 0.5).astype(int).ravel() == y).mean()

## 朴素训练循环（没有护栏）

先跑一个"裸"循环，看它的问题：没有固定 seed（不可复现）、没有 checkpoint（断了重来）、固定学习率（收敛慢）、没有早停（过拟合风险）。

In [ ]:
def train_naive(net, X_tr, y_tr, X_va, y_va, steps=1500, lr=0.5):
    log = []
    for step in range(steps):
        p = net.forward(X_tr)
        loss = -np.mean(y_tr * np.log(p + 1e-9) + (1 - y_tr) * np.log(1 - p + 1e-9))
        gW1, gb1, gW2, gb2 = net.backward(X_tr, y_tr, p)
        net.W1 -= lr * gW1; net.b1 -= lr * gb1
        net.W2 -= lr * gW2; net.b2 -= lr * gb2
        if step % 300 == 0:
            log.append((step, loss, acc(net, X_va, y_va)))
    return log

net0 = Net(seed=0)
log0 = train_naive(net0, X_tr, y_tr, X_va, y_va)
for row in log0:
    print(f"step {row[0]:4d}  loss = {row[1]:.4f}  验证准确率 = {row[2]:.4f}")
print("最终验证准确率 =", round(acc(net0, X_va, y_va), 4))

## 每个护栏的原理

**1) seed 管理**：`np.random.default_rng(seed)` 让权重初始化可复现。工程上还会固定数据打乱顺序、框架级 seed。复现是实验管理的底线（04 课展开）。

**2) checkpoint**：每个 epoch 把"当前最优参数"存盘：

$$\text{best.npz} = \{W_1, b_1, W_2, b_2\} \quad \text{（验证集最优时覆盖）}$$

中断后从磁盘恢复参数继续，不用从头跑。

**3) early stopping**：设 `patience`，验证指标连续 N 步不创新高就停。数学直觉：验证集损失先降后升的拐点，就是"开始过拟合"的位置。

**4) 学习率调度**：线性衰减 $\eta_t = \eta_0 \cdot (1 - t/T)$，或余弦退火、warmup+decay。前期大步跳过坏区域，后期小步落进好极小值。

**5) 梯度裁剪**：若全参数梯度的 L2 范数超阈值，等比例缩放：

$$g \leftarrow g \cdot \min\left(1,\ \frac{c}{\|g\|_2}\right)$$

防止个别大梯度一步把参数"冲飞"（RNN/LLM 里尤其致命）。

In [ ]:
def clip_grads(grads, max_norm=1.0):
    """按全局 L2 范数裁剪梯度。"""
    total = np.sqrt(sum(float((g ** 2).sum()) for g in grads))
    if total > max_norm:
        s = max_norm / (total + 1e-12)
        return [g * s for g in grads]
    return grads

def train_guarded(net, X_tr, y_tr, X_va, y_va, steps=1500, lr0=0.5,
                  patience=200, max_norm=1.0, ckpt="best.npz"):
    """带全部护栏的训练循环：LR 调度 + 梯度裁剪 + 早停 + checkpoint。"""
    best_val, wait = -1.0, 0
    log = []
    for step in range(steps):
        lr = lr0 * (1 - step / steps)                 # 1) 线性 LR 调度
        p = net.forward(X_tr)
        loss = -np.mean(y_tr * np.log(p + 1e-9) + (1 - y_tr) * np.log(1 - p + 1e-9))
        grads = net.backward(X_tr, y_tr, p)
        grads = clip_grads(grads, max_norm)           # 2) 梯度裁剪
        for w, g in zip(["W1", "b1", "W2", "b2"], grads):
            setattr(net, w, getattr(net, w) - lr * g)
        va = acc(net, X_va, y_va)
        if va > best_val:                             # 3) 早停 + checkpoint
            best_val, wait = va, 0
            np.savez(ckpt, **net.params())
        else:
            wait += 1
        if step % 300 == 0:
            log.append((step, loss, va))
        if wait >= patience:
            print(f"step {step}: 早停触发（验证集连续 {patience} 步未提升）")
            break
    return log, best_val

net1 = Net(seed=0)
log1, best1 = train_guarded(net1, X_tr, y_tr, X_va, y_va)
for row in log1:
    if len(row) == 3:
        print(f"step {row[0]:4d}  loss = {row[1]:.4f}  验证准确率 = {row[2]:.4f}")
print("最佳验证准确率 =", round(best1, 4))

# 曲线对比：朴素 vs 带护栏
plt.figure(figsize=(6.5, 4))
plt.plot([r[0] for r in log0], [r[2] for r in log0], "o-", label="朴素（固定 LR）")
plt.plot([r[0] for r in log1 if len(r) == 3], [r[2] for r in log1 if len(r) == 3], "s--", label="带护栏（LR 调度+早停）")
plt.xlabel("step"); plt.ylabel("验证准确率")
plt.title("训练护栏的效果：更稳、更早收敛、防过拟合")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# 断点续训验证：新建网络，从 checkpoint 恢复，预测必须与保存时一致
net2 = Net(seed=999)          # 故意用不同 seed
saved = np.load("best.npz")
net2.load({k: saved[k] for k in saved.files})
print("恢复后验证准确率 =", round(acc(net2, X_va, y_va), 4), "（应等于上面的最佳值）")
assert abs(acc(net2, X_va, y_va) - best1) < 1e-6, "checkpoint 恢复不一致！"
print("checkpoint 断点续训验证通过：参数、状态可完整恢复")

## 进阶：混合精度与多卡（概念层）

**混合精度 AMP**：权重用 fp32 存，计算用 fp16 跑，梯度回传前缩放到 fp32。收益：显存减半、速度 1.5~3×。代价：小梯度可能下溢 → 用 loss scaling 补偿。公式化：前向 $y = W_{fp32}$ 但矩阵乘在 fp16 下执行，$\text{loss} \cdot s$ 放大后回传再除回。

**多卡 DDP（分布式数据并行）**：每张卡一份模型副本、各吃一个数据分片，每步同步梯度再更新：

$$g = \frac{1}{K} \sum_{k=1}^{K} g_k \quad \text{（All-Reduce 平均梯度）}$$

通信量 = 参数大小 ×2（每步）。这就是为什么参数越大越需要梯度压缩/延迟同步等技术。用伪代码理解：

```python
# 伪代码：DDP 每步
for step in range(steps):
    pred = model(data_chunk_k)            # 每卡自己的数据
    loss = criterion(pred, label_k)
    loss.backward()
    all_reduce(grads, op=MEAN)            # 关键：跨卡求平均
    optimizer.step()
```

工程要点：batch size 要随卡数放大（global batch = K × per-gpu batch），学习率通常也要相应放大。

## 课后练习

1. **代码练习**：把线性 LR 调度换成余弦退火 $\eta_t = \eta_{min} + (\eta_0 - \eta_{min})\cdot\frac{1+\cos(\pi t/T)}{2}$，重跑对比曲线——为什么余弦退火在 Transformer 训练里是默认选择？
2. **断点练习**：训练到一半人为"断电"（把循环改成 500 步后 break），用 checkpoint 恢复并继续完成剩余步数，验证最终准确率与一口气训练一致。
3. **Kaggle：Digit Recognizer**（<https://www.kaggle.com/c/digit-recognizer>）：把本课护栏思想用在 MNIST 上——用 sklearn 的 `MLPClassifier`（或自己写训练循环）训练，目标提交准确率 **≥ 0.96**；练习中注意记录你的超参与结果（衔接 04 课）。